# Redes Neurais Convolucionais

Uma MLP achata a imagem em um vetor e perde a noção de vizinhança entre pixels. A **convolução** resolve isso: um filtro pequeno desliza pela imagem, e o mesmo conjunto de pesos é usado em todas as posições.

Vamos ver como imagens são representadas, fazer uma convolução à mão, usar a camada `nn.Conv2d`, comparar uma CNN com uma MLP no MNIST e, por fim, montar uma CNN para o CIFAR-10.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Representação de imagens

Para a rede, uma imagem é um tensor de números. Uma imagem em tons de cinza é uma matriz $H \times W$ com a intensidade de cada pixel. O `transforms.ToTensor()` converte os valores de 0 a 255 para o intervalo $[0, 1]$, em que 0 é preto e 1 é branco.

No PyTorch, as imagens têm um eixo de **canais** antes da altura e da largura: formato `[C, H, W]`. No MNIST, $C = 1$.

In [ ]:
mnist = datasets.MNIST(root="data", train=True, download=True, transform=transforms.ToTensor())

digit, label = mnist[0]
print(digit.shape)   # [1, 28, 28]
print(label)

In [ ]:
plt.imshow(digit[0], cmap="gray")
plt.title(f"rótulo: {label}")
plt.show()

Cada pixel é só um número. Um pedaço da imagem, no meio do traço:

In [ ]:
torch.set_printoptions(precision=1)
print(digit[0, 10:16, 10:16])

Uma imagem colorida tem três canais, um para cada cor: vermelho, verde e azul (**RGB**). A cor de um pixel vem da combinação dos três valores. Esta imagem é do CIFAR-10, que vamos usar no fim da aula.

In [ ]:
cifar = datasets.CIFAR10(root="data", train=True, download=True, transform=transforms.ToTensor())

photo, _ = cifar[7]
print(photo.shape)   # [3, 32, 32]

O matplotlib espera os canais no fim, `[H, W, C]`, então usamos `permute` para mudar a ordem dos eixos.

In [ ]:
plt.imshow(photo.permute(1, 2, 0))   # [3, 32, 32] -> [32, 32, 3]
plt.show()

Cada canal, sozinho, é uma imagem em tons de cinza. Quanto mais claro o pixel, mais daquela cor ele tem: o cavalo marrom é mais claro no canal vermelho do que no azul.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(photo[0], cmap="gray", vmin=0, vmax=1)
axes[0].set_title("vermelho")

axes[1].imshow(photo[1], cmap="gray", vmin=0, vmax=1)
axes[1].set_title("verde")

axes[2].imshow(photo[2], cmap="gray", vmin=0, vmax=1)
axes[2].set_title("azul")

plt.show()

Para treinar, as imagens são agrupadas em lotes com formato `[N, C, H, W]`, em que $N$ é o tamanho do lote. Uma MLP precisa achatar cada imagem em um vetor e perde a informação de quais pixels são vizinhos. A convolução trabalha direto no formato `[N, C, H, W]`.

In [ ]:
images, labels = next(iter(DataLoader(mnist, batch_size=16)))

print(images.shape)              # [16, 1, 28, 28]
print(images.flatten(1).shape)   # [16, 784], o que a MLP recebe

## Convolução feita à mão

Um **kernel** $W$ de tamanho $K \times K$ desliza pela imagem $X$. Em cada posição, a saída é a soma dos produtos entre o kernel e a janela da imagem embaixo dele:

$$
Y[i, j] = \sum_{m=0}^{K-1} \sum_{n=0}^{K-1} X[i + m, \, j + n] \, W[m, n]
$$

A saída $Y$ é chamada de **mapa de características**. Sem preenchimento nas bordas, ela tem tamanho $(H - K + 1) \times (W - K + 1)$.

In [ ]:
def convolve(image, kernel):
    K = kernel.shape[0]
    H, W = image.shape
    output = torch.zeros(H - K + 1, W - K + 1)

    for i in range(H - K + 1):
        for j in range(W - K + 1):
            window = image[i:i + K, j:j + K]
            output[i, j] = (window * kernel).sum()

    return output

O kernel funciona como um detector de padrão. O filtro de Sobel abaixo responde a bordas verticais: positivo onde a imagem passa de escuro para claro, negativo no contrário.

In [ ]:
sobel = torch.tensor([[-1., 0., 1.],
                      [-2., 0., 2.],
                      [-1., 0., 1.]])

edges = convolve(digit[0], sobel)        # [26, 26]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
ax1.imshow(digit[0], cmap="gray")
ax1.set_title("entrada")
ax2.imshow(edges, cmap="RdBu_r")
ax2.set_title("convolução com Sobel")
plt.show()

Em uma CNN, ninguém escolhe os valores do kernel à mão: eles são parâmetros aprendidos pelo gradiente, como os pesos de uma `nn.Linear`.

## A camada `nn.Conv2d`

A `nn.Conv2d` faz a mesma operação, com os argumentos:

- `in_channels`: canais da entrada (1 para tons de cinza, 3 para RGB);
- `out_channels`: quantos kernels aplicar, ou seja, quantos mapas de saída;
- `kernel_size`: o tamanho $K$ do kernel;
- `padding`: zeros acrescentados nas bordas;
- `stride`: o passo do deslizamento.

A entrada tem formato `[N, C, H, W]` e o tamanho da saída é

$$
H_{out} = \left\lfloor \frac{H_{in} + 2 \cdot \text{padding} - K}{\text{stride}} \right\rfloor + 1
$$

Primeiro, conferimos que ela dá o mesmo resultado da nossa função, colocando o Sobel nos pesos:

In [ ]:
conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, bias=False)
print(conv.weight.shape)                  # [out_channels, in_channels, K, K]

with torch.no_grad():
    conv.weight[0, 0] = sobel
    output = conv(digit[None])            # [1, 1, 28, 28] -> [1, 1, 26, 26]

print(torch.allclose(output[0, 0], edges, atol=1e-5))

Agora, o efeito de cada argumento no formato da saída. Com `kernel_size=3` e `padding=1`, a imagem mantém o tamanho; com `stride=2`, ela cai pela metade. O **max pooling** também reduz pela metade, pegando o maior valor de cada janela $2 \times 2$.

In [ ]:
x = torch.randn(8, 1, 28, 28)   # lote de 8 imagens

print(nn.Conv2d(1, 16, kernel_size=3)(x).shape)                       # [8, 16, 26, 26]
print(nn.Conv2d(1, 16, kernel_size=3, padding=1)(x).shape)            # [8, 16, 28, 28]
print(nn.Conv2d(1, 16, kernel_size=3, padding=1, stride=2)(x).shape)  # [8, 16, 14, 14]
print(nn.MaxPool2d(2)(x).shape)                                       # [8, 1, 14, 14]

## CNN no MNIST

Os dados, e as funções de treino e de avaliação, iguais às dos materiais anteriores.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_set = datasets.MNIST(root="data", train=True, download=True, transform=transform)
test_set = datasets.MNIST(root="data", train=False, download=True, transform=transform)

train_dataloader = DataLoader(train_set, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_set, batch_size=500)

In [ ]:
criterion = nn.CrossEntropyLoss()


def accuracy(model, dataloader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            correct += (model(images).argmax(dim=1) == labels).sum().item()
    return correct / len(dataloader.dataset)


def train(model, train_dataloader, test_dataloader, epochs, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for images, labels in train_dataloader:
            images, labels = images.to(device), labels.to(device)
            loss = criterion(model(images), labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        mean_loss = total_loss / len(train_dataloader)
        test_accuracy = accuracy(model, test_dataloader)
        print(f"[{epoch + 1}/{epochs}] perda {mean_loss:.4f}, acurácia de teste {test_accuracy:.3f}")

    return test_accuracy


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

### MLP

Para ter uma referência, primeiro a MLP dos materiais anteriores, com uma camada oculta de 128 unidades.

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),                 # [N, 784]
            nn.Linear(28 * 28, 128),      # [N, 128]
            nn.ReLU(),
            nn.Linear(128, 10),           # [N, 10]
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
torch.manual_seed(42)
mlp = MLP().to(device)
mlp_accuracy = train(mlp, train_dataloader, test_dataloader, epochs=5)

### CNN

A rede tem duas partes: `features`, com blocos de convolução, ReLU e pooling, e `classifier`, que achata os mapas e aplica uma camada linear. Os comentários mostram o formato da saída de cada camada.

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),    # [N, 16, 28, 28]
            nn.ReLU(),
            nn.MaxPool2d(2),                               # [N, 16, 14, 14]
            nn.Conv2d(16, 32, kernel_size=3, padding=1),   # [N, 32, 14, 14]
            nn.ReLU(),
            nn.MaxPool2d(2),                               # [N, 32, 7, 7]
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                                  # [N, 32 * 7 * 7]
            nn.Linear(32 * 7 * 7, 10),                     # [N, 10]
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
torch.manual_seed(42)
cnn = CNN().to(device)
cnn_accuracy = train(cnn, train_dataloader, test_dataloader, epochs=5)

### Comparação

In [ ]:
for name, model, acc in [("MLP", mlp, mlp_accuracy), ("CNN", cnn, cnn_accuracy)]:
    print(f"{name}: {count_parameters(model):7,} parâmetros, acurácia de teste {acc:.3f}, "
          f"{int(round((1 - acc) * len(test_set)))} erros")

Com o mesmo treino, a CNN erra bem menos que a MLP e usa cerca de um quinto dos parâmetros. A MLP precisa de um peso para cada pixel em cada unidade; a CNN reaproveita os mesmos kernels pequenos em todas as posições.

## CNN no CIFAR-10

O **CIFAR-10** tem fotos coloridas de $32 \times 32$ pixels em dez classes. Cada imagem tem 3 canais, então a primeira convolução recebe `in_channels=3`. Complete as células com `...`.

In [ ]:
classes = ["avião", "automóvel", "pássaro", "gato", "cervo",
           "cachorro", "sapo", "cavalo", "navio", "caminhão"]

cifar_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

cifar_train_set = datasets.CIFAR10(root="data", train=True, download=True, transform=cifar_transform)
cifar_test_set = datasets.CIFAR10(root="data", train=False, download=True, transform=cifar_transform)

cifar_train_dataloader = DataLoader(cifar_train_set, batch_size=128, shuffle=True)
cifar_test_dataloader = DataLoader(cifar_test_set, batch_size=500)

images, labels = next(iter(cifar_train_dataloader))
print(images.shape)   # [128, 3, 32, 32]

fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
for ax, image, label in zip(axes, images, labels):
    ax.imshow((image * 0.5 + 0.5).permute(1, 2, 0))   # [3, H, W] -> [H, W, 3]
    ax.set_title(classes[label])
    ax.axis("off")
plt.show()

Monte a rede abaixo. Cada bloco é `Conv2d` com `kernel_size=3` e `padding=1`, seguida de `ReLU` e `MaxPool2d(2)`.

| camada | formato da saída |
|---|---|
| entrada | $N \times 3 \times 32 \times 32$ |
| bloco com 32 canais | $N \times 32 \times 16 \times 16$ |
| bloco com 64 canais | $N \times 64 \times 8 \times 8$ |
| bloco com 128 canais | $N \times 128 \times 4 \times 4$ |
| `Flatten` | ? |
| `Linear` | $N \times 10$ |

In [ ]:
class CIFARCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            ...
        )
        self.classifier = nn.Sequential(
            ...
        )

    def forward(self, x):
        return self.classifier(self.features(x))

Confira os formatos antes de treinar:

In [ ]:
print(CIFARCNN()(images).shape)   # deve ser [128, 10]

Treine a rede por 10 épocas usando a função `train`.

In [ ]:
torch.manual_seed(42)
cifar_model = ...
...

Por fim, tente melhorar a acurácia: mais canais, duas convoluções por bloco, `nn.BatchNorm2d` depois de cada convolução, ou `nn.Dropout` antes da camada linear.